# Bayan wake-word — Colab trainer (faithful port of openWakeWord automatic training)

This is a direct port of openWakeWord's own end-to-end-tested
[`automatic_model_training.ipynb`](https://github.com/dscripka/openWakeWord/blob/main/notebooks/automatic_model_training.ipynb):
same single-source install, same pinned dependencies, same `custom_model.yml` config, same
`train.py` steps. It only adds: (a) upload your recordings ZIP, (b) inject your real clips into the
positive set before generation, (c) export `bayan_wake.onnx` + `wake_meta.json` + a downloadable ZIP.

**You do nothing but:** Runtime → Change runtime type → **GPU (T4)** → **Runtime → Run all** →
upload `wake_data_v1-egyptian-pilot.zip` when asked. No code edits, no manual installs.

> openWakeWord's automatic training is **Linux-only** (Piper TTS) — Colab is Linux, so this is fine.
> Full run ≈ 1–1.5 h on a free T4 (data download + synthetic generation + training).

In [ ]:
## Environment setup — installs everything (verbatim from openWakeWord's official notebook)

# piper-sample-generator (synthetic speech for training clips; Linux only)
!git clone https://github.com/rhasspy/piper-sample-generator
!wget -O piper-sample-generator/models/en_US-libritts_r-medium.pt 'https://github.com/rhasspy/piper-sample-generator/releases/download/v2.0.0/en_US-libritts_r-medium.pt'
!pip install piper-phonemize
!pip install webrtcvad

# openwakeword — SINGLE source: clone + editable install (do NOT also pip-install the package)
!git clone https://github.com/dscripka/openwakeword
!pip install -e ./openwakeword

# training dependencies (pinned — includes torchinfo + speechbrain)
!pip install mutagen==1.47.0
!pip install torchinfo==1.8.0
!pip install torchmetrics==1.2.0
!pip install speechbrain==0.5.14
!pip install audiomentations==0.33.0
!pip install torch-audiomentations==0.11.0
!pip install acoustics==0.2.6
!pip install tensorflow-cpu==2.8.1
!pip install tensorflow_probability==0.16.0
!pip install onnx_tf==1.10.0
!pip install pronouncing==0.2.0
!pip install datasets==2.14.6
!pip install deep-phonemizer==0.0.19

# Required feature models (Colab workaround)
import os
os.makedirs("./openwakeword/openwakeword/resources/models", exist_ok=True)
!wget https://github.com/dscripka/openWakeWord/releases/download/v0.5.1/embedding_model.onnx -O ./openwakeword/openwakeword/resources/models/embedding_model.onnx
!wget https://github.com/dscripka/openWakeWord/releases/download/v0.5.1/embedding_model.tflite -O ./openwakeword/openwakeword/resources/models/embedding_model.tflite
!wget https://github.com/dscripka/openWakeWord/releases/download/v0.5.1/melspectrogram.onnx -O ./openwakeword/openwakeword/resources/models/melspectrogram.onnx
!wget https://github.com/dscripka/openWakeWord/releases/download/v0.5.1/melspectrogram.tflite -O ./openwakeword/openwakeword/resources/models/melspectrogram.tflite

In [ ]:
import os
import numpy as np
import torch
import sys
from pathlib import Path
import uuid
import yaml
import datasets
import scipy
import scipy.io.wavfile
from tqdm import tqdm

In [ ]:
# Upload your recordings ZIP (from the recorder tool). Run-all will pause here for the file dialog.
from google.colab import files
import zipfile, glob
print("Choose wake_data_v1-egyptian-pilot.zip ...")
up = files.upload()
zname = [f for f in up if f.endswith(".zip")][0]
zipfile.ZipFile(zname).extractall(".")
real_clips = sorted(glob.glob("wake_data/positive/**/*.wav", recursive=True))
print("real positive recordings found:", len(real_clips))
assert len(real_clips) > 0, "No positives found under wake_data/positive in the uploaded ZIP."


In [ ]:
# Download room impulse responses collected by MIT
# https://mcdermottlab.mit.edu/Reverb/IR_Survey.html

output_dir = "./mit_rirs"
if not os.path.exists(output_dir):
    os.mkdir(output_dir)
rir_dataset = datasets.load_dataset("davidscripka/MIT_environmental_impulse_responses", split="train", streaming=True)

# Save clips to 16-bit PCM wav files
for row in tqdm(rir_dataset):
    name = row['audio']['path'].split('/')[-1]
    scipy.io.wavfile.write(os.path.join(output_dir, name), 16000, (row['audio']['array']*32767).astype(np.int16))

In [ ]:
## Download noise and background audio (Audioset part + FMA small)

if not os.path.exists("audioset"):
    os.mkdir("audioset")

fname = "bal_train09.tar"
out_dir = f"audioset/{fname}"
link = "https://huggingface.co/datasets/agkphysics/AudioSet/resolve/main/data/" + fname
!wget -O {out_dir} {link}
!cd audioset && tar -xvf bal_train09.tar

output_dir = "./audioset_16k"
if not os.path.exists(output_dir):
    os.mkdir(output_dir)

audioset_dataset = datasets.Dataset.from_dict({"audio": [str(i) for i in Path("audioset/audio").glob("**/*.flac")]})
audioset_dataset = audioset_dataset.cast_column("audio", datasets.Audio(sampling_rate=16000))
for row in tqdm(audioset_dataset):
    name = row['audio']['path'].split('/')[-1].replace(".flac", ".wav")
    scipy.io.wavfile.write(os.path.join(output_dir, name), 16000, (row['audio']['array']*32767).astype(np.int16))

output_dir = "./fma"
if not os.path.exists(output_dir):
    os.mkdir(output_dir)
fma_dataset = datasets.load_dataset("rudraml/fma", name="small", split="train", streaming=True)
fma_dataset = iter(fma_dataset.cast_column("audio", datasets.Audio(sampling_rate=16000)))

n_hours = 1  # recommend increasing for full-scale training
for i in tqdm(range(n_hours*3600//30)):  # FMA clips are 30 s each
    row = next(fma_dataset)
    name = row['audio']['path'].split('/')[-1].replace(".mp3", ".wav")
    scipy.io.wavfile.write(os.path.join(output_dir, name), 16000, (row['audio']['array']*32767).astype(np.int16))
    i += 1
    if i == n_hours*3600//30:
        break

In [ ]:
# Download pre-computed openWakeWord features for training + validation
# training set (~2,000 hours, ACAV100M) and false-positive validation set (~11 hours)
!wget https://huggingface.co/datasets/davidscripka/openwakeword_features/resolve/main/openwakeword_features_ACAV100M_2000_hrs_16bit.npy
!wget https://huggingface.co/datasets/davidscripka/openwakeword_features/resolve/main/validation_set_features.npy

In [ ]:
# Load the OFFICIAL config template (has piper_sample_generator_path + all required keys), then override.
MODEL_VERSION   = "bayan-wake-v1-egyptian"
DATASET_VERSION = "v1-egyptian-pilot"

config = yaml.load(open("openwakeword/examples/custom_model.yml").read(), yaml.Loader)
config["target_phrase"] = ["bayan", "beyan", "bayaan"]   # English spellings Piper can voice; your real Arabic clips add the rest
config["model_name"]    = "bayan"
config["n_samples"]     = 5000      # synthetic positives (topped up on top of your real clips)
config["n_samples_val"] = 1000
config["steps"]         = 15000
config["target_accuracy"] = 0.6
config["target_recall"]   = 0.25
config["background_paths"] = ["./audioset_16k", "./fma"]
config["false_positive_validation_data_path"] = "validation_set_features.npy"
config["feature_data_files"] = {"ACAV100M_sample": "openwakeword_features_ACAV100M_2000_hrs_16bit.npy"}
# piper_sample_generator_path stays "./piper-sample-generator" (from the template) — matches the clone above.
with open("bayan.yaml", "w") as f:
    yaml.dump(config, f)
print("piper:", config["piper_sample_generator_path"], "| output_dir:", config["output_dir"], "| model:", config["model_name"])

In [ ]:
# Put your real clips into the positive dirs so generation tops up around them and augmentation includes them.
# (train.py counts existing files and only generates the remainder; --augment_clips globs every *.wav here.)
import shutil
base = os.path.join(config["output_dir"], config["model_name"])
ptr = os.path.join(base, "positive_train"); pte = os.path.join(base, "positive_test")
os.makedirs(ptr, exist_ok=True); os.makedirs(pte, exist_ok=True)

def place(src, dst, min_secs=1.5, sr=16000):
    r, dat = scipy.io.wavfile.read(src)
    if getattr(dat, "ndim", 1) > 1: dat = dat[:, 0]
    if r != sr:  # recorder already exports 16 kHz; this is just a safety net
        dat = np.interp(np.linspace(0, len(dat), int(len(dat)*sr/r), endpoint=False),
                        np.arange(len(dat)), dat)
    dat = dat.astype(np.int16)
    need = int(min_secs*sr)
    if len(dat) < need:  # pad short wake clips so the feature window is satisfied
        dat = np.concatenate([dat, np.zeros(need-len(dat), dtype=np.int16)])
    scipy.io.wavfile.write(dst, sr, dat)

for i, p in enumerate(real_clips):
    dst_dir = pte if (i % 6 == 0) else ptr   # hold ~1/6 out for validation
    place(p, os.path.join(dst_dir, f"real_{i:03d}.wav"))
print("real clips injected -> positive_train:", len(os.listdir(ptr)), "| positive_test:", len(os.listdir(pte)))

In [ ]:
# Step 1: generate synthetic clips (tops up to n_samples around your real clips). ~10-20 min on a T4.
!{sys.executable} openwakeword/openwakeword/train.py --training_config bayan.yaml --generate_clips

In [ ]:
# Step 2: augment the clips (real + synthetic).
!{sys.executable} openwakeword/openwakeword/train.py --training_config bayan.yaml --augment_clips

In [ ]:
# Step 3: train the model (auto-exports .onnx + .tflite when done).
!{sys.executable} openwakeword/openwakeword/train.py --training_config bayan.yaml --train_model

In [ ]:
# Package: bayan_wake.onnx + the shared feature models + versioned wake_meta.json -> downloadable ZIP.
import shutil, json, hashlib, glob
cands = glob.glob(os.path.join(config["output_dir"], "**", "bayan*.onnx"), recursive=True)
assert cands, "No bayan*.onnx produced — read the Step-3 output above for the training error."
onnx_src = max(cands, key=os.path.getsize)
os.makedirs("bayan-wake-out", exist_ok=True)
shutil.copy(onnx_src, "bayan-wake-out/bayan_wake.onnx")
onnx_sha = hashlib.sha256(open("bayan-wake-out/bayan_wake.onnx", "rb").read()).hexdigest()

# openWakeWord inference is a CHAIN: melspectrogram.onnx -> embedding_model.onnx -> this classifier.
# Bundle the two shared feature models so the browser has the full pipeline.
res = "openwakeword/openwakeword/resources/models"
for m in ["melspectrogram.onnx", "embedding_model.onnx"]:
    if os.path.exists(os.path.join(res, m)): shutil.copy(os.path.join(res, m), "bayan-wake-out/"+m)

ds_sha = None
if os.path.exists("wake_data/recording_manifest.json"):
    ds_sha = hashlib.sha256(open("wake_data/recording_manifest.json", "rb").read()).hexdigest()

meta = {
  "modelVersion": MODEL_VERSION, "datasetVersion": DATASET_VERSION,
  "trainer": "openWakeWord automatic_model_training (faithful port)",
  "targetPhrase": config["target_phrase"], "wakePhrases": ["Bayan", "بيان", "يا بيان"],
  "sampleRate": 16000, "inferenceChain": ["melspectrogram.onnx", "embedding_model.onnx", "bayan_wake.onnx"],
  "onnxSource": os.path.relpath(onnx_src), "onnxSha256": onnx_sha, "datasetManifestSha256": ds_sha,
  "realPositives": len(real_clips), "syntheticPositives": config["n_samples"], "steps": config["steps"],
  # Fill from a real measurement (see the eval note below); never invent numbers:
  "metrics": {"quietDetection": None, "noisyDetection": None, "falseActivationsPerHour": None, "p95LatencyMs": None},
}
open("bayan-wake-out/wake_meta.json", "w").write(json.dumps(meta, ensure_ascii=False, indent=2))
print(json.dumps(meta, ensure_ascii=False, indent=2))

zip_name = f"bayan-wake-model_{MODEL_VERSION}"
shutil.make_archive(zip_name, "zip", "bayan-wake-out")
from google.colab import files as F
F.download(zip_name + ".zip")
print("DONE ->", zip_name + ".zip  (bayan_wake.onnx + melspectrogram.onnx + embedding_model.onnx + wake_meta.json)")

## After it finishes
You get `bayan-wake-model_bayan-wake-v1-egyptian.zip` containing `bayan_wake.onnx`,
the shared `melspectrogram.onnx` + `embedding_model.onnx`, and `wake_meta.json`. Send it back.

**Honest eval note (single-speaker pilot):** this v1 was trained on one speaker's 24 real clips plus
synthetic data. On-speaker detection will look high but does NOT prove cross-speaker performance, and
false-activations/hour must be MEASURED against real long negative audio — those numbers come from the
deployed benchmark, not from training. Leave `metrics` null until measured; never invent them.